In [1]:
import pandas as pd
import re

# --- 1. Load and combine Data ---
df_dict = pd.read_excel("../data/raw/online_retail_II.xlsx", sheet_name = None)
df = pd.concat(df_dict.values(), ignore_index = True)

print(f"Before cleaning: {df.shape}")

Before cleaning: (1067371, 8)


In [2]:
# --- 2. Data Profiling ---

# a.Check unique countries to spot typos
unique_countries = df['Country'].value_counts()
print(f"Total unique countries: {len(unique_countries)}")

# b.Filter non-numeric stock codes from RAW data
non_numeric_stock = df[~df['StockCode'].astype(str).str.isnumeric()]
unique_non_numeric = non_numeric_stock['StockCode'].astype(str).str.strip().unique()

valid_variants = []
suspicious_codes = []

for code in unique_non_numeric:
    # If code is numbers followed by ONE letter (e.g. , 84406B), it's valid
    if re.match(r'^\d+[A-Za-z]$', code):
        valid_variants.append(code)
    else:
        suspicious_codes.append(code)
        
print(f"Total unique non-numeric StockCodes codes: {len(unique_non_numeric)}")
print(f"Valid variants: {len(valid_variants)}")
print(f"Suspicious codes: {len(suspicious_codes)}")
print("\nList of suspicious codes:")
print(suspicious_codes)

# c. Profile Customer ID (Corrected version)
non_numeric_mask = pd.to_numeric(df['Customer ID'], errors= 'coerce').isna() & df['Customer ID'].notna()
weird_ids = df.loc[non_numeric_mask, 'Customer ID']
print(f"\nNumber of wierd (non-numeric) Customer IDs: {len(weird_ids)}")
if len(weird_ids)>0:
    print("Examples:",weird_ids.unique()[:10])

print("\nValue counts for suspicious codes:")
print(non_numeric_stock[non_numeric_stock['StockCode'].isin(suspicious_codes)]['StockCode'].value_counts())
print("Full list of suspicious codes:")
print(suspicious_codes)

Total unique countries: 43
Total unique non-numeric StockCodes codes: 1714
Valid variants: 1647
Suspicious codes: 67

List of suspicious codes:
['POST', '79323LP', '15056BL', 'D', '79323GR', 'DCGS0058', 'DCGS0068', 'DOT', '15056bl', 'M', 'DCGS0004', 'DCGS0076', 'C2', 'BANK CHARGES', 'DCGS0003', 'TEST001', 'gift_0001_80', 'DCGS0072', 'gift_0001_20', 'DCGS0044', 'TEST002', 'gift_0001_10', 'gift_0001_50', 'DCGS0066N', 'gift_0001_30', 'PADS', '72024HC', 'ADJUST', 'gift_0001_40', 'gift_0001_60', 'gift_0001_70', 'gift_0001_90', 'DCGSSGIRL', 'DCGS0006', 'DCGS0016', 'DCGS0027', 'DCGS0036', 'DCGS0039', 'DCGS0060', 'DCGS0056', 'DCGS0059', 'GIFT', 'DCGSLBOY', 'm', 'DCGS0053', 'DCGS0062', 'DCGS0037', 'DCGSSBOY', 'DCGSLGIRL', 'S', 'DCGS0069', 'DCGS0070', 'DCGS0075', 'B', 'DCGS0041', 'ADJUST2', 'C3', 'SP1002', 'AMAZONFEE', 'DCGS0055', 'DCGS0074', 'DCGS0057', 'DCGS0073', 'DCGS0071', 'DCGS0066P', 'DCGS0067', 'CRUK']

Number of wierd (non-numeric) Customer IDs: 0

Value counts for suspicious codes:
Sto

In [3]:
# --- 3. Remove Invalid Data (Returns, Missing IDs, Zero Prices) ---
# Remove rows without Customer ID (essential for RFM)
df = df.dropna(subset = ['Customer ID'])
print(f"After removing missing Customer ID: {df.shape}")

# Remove returns (negative / zero Quantity) for sales analysis
df = df[df['Quantity'] > 0]
print(f"After removing returns: {df.shape}")

#Remove cancelled invoices (Invoice IDs starting with 'C')
df = df[~df['Invoice'].astype(str).str.startswith('C')]

#Remove zero / negative prices
df = df[df['Price']>0]

# Remove rows without StockCode
df = df.dropna(subset = ['StockCode'])

# a.Remove non-product StockCodes (fees, adjustments, tests)
codes_to_drop = ['POST', 'DOT', 'M', 'm', 'D', 'S', 'B', 'C2', 'C3', 'BANK CHARGES', 'AMAZONFEE', 'CRUK', 'ADJUST', 'ADJUST2', 'TEST001', 'TEST002', 'GIFT', 'PADS', 'SP1002']
df = df[~df['StockCode'].isin(codes_to_drop)]

# b.Remove DCGS and gift codes using startswith
df = df[~df['StockCode'].astype(str).str.startswith(('DCGS', 'gift_'))]

print(f"After removing non-product codes: {df.shape}")

After removing missing Customer ID: (824364, 8)
After removing returns: (805620, 8)
After removing non-product codes: (802632, 8)


In [4]:
#Create TotalPrice
df['TotalPrice'] = df['Quantity'] * df['Price']

In [5]:
#---4. Advanced Data Cleaning (Duplicates, Text, Dates)  ---

#Check for duplicates
print(f"Duplicate rows before: {df.duplicated().sum()}")

df = df.drop_duplicates()
print(f"Duplicate rows after: {df.shape}")

#Clean text columns (strip whitespace and standardize case)
text_columns = ['Description','Country']
for col in text_columns:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()
        if col == 'Country':
            df[col] = df[col].str.title()

# Clean StockCode (remove whitespace, uppercase)
df['StockCode'] = df['StockCode'].astype(str).str.strip().str.upper()

# Convert Customer ID to string (remove .0 if it exists)
df['Customer ID'] = df['Customer ID'].astype(str).str.replace('.0','',regex=False)

#Ensure InvoiceDate is datetime (Crucial for RFM)
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

Duplicate rows before: 26055
Duplicate rows after: (776577, 9)


In [7]:
# --- 5. Check for outliers and Save ---
print("\nPrice 99th percentile:", df['Price'].quantile(0.99))
print("Max Price:",df['Price'].max())


Price 99th percentile: 12.75
Max Price: 649.5


In [8]:
#Ensure InvoiceDate is datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

#Save cleaned data
df.to_csv("../data/processed/online_retail_cleaned.csv", index = False)
print(f" Final cleaned data Saved to data/processed/online_retail_cleaned.csv successfully! Shape: {df.shape}")

 Final cleaned data Saved to data/processed/online_retail_cleaned.csv successfully! Shape: (776577, 9)
